# 01 — Data Exploration (EDA)
Real MovieLens data only -- this project trains and evaluates exclusively on `ml-32m` (32,000,204 ratings). Run `python scripts/download_dataset.py` first if `data/raw/ml-32m/` is empty.

**Heads up:** this notebook operates on the full 32M-row dataset now, not the old 100K-row dev sample -- cells that materialize the whole ratings frame (not the chunked reader) need a machine/session with enough RAM (16GB+ recommended).
No synthetic rows are introduced anywhere in this notebook.

In [ ]:
import os, sys
sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src.data import loader

ratings = loader.load_ratings()
movies = loader.load_movies()
tags = loader.load_tags()
links = loader.load_links()
print(f"ratings={len(ratings):,}  movies={len(movies):,}  tags={len(tags):,}  links={len(links):,}")


## Rating distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=ax, color='#E3B23C')
ax.set_title('Rating value distribution (real MovieLens ratings)')
ax.set_xlabel('rating'); ax.set_ylabel('count')
plt.tight_layout(); plt.show()
print(ratings['rating'].describe())


## Ratings per user (long-tail check)

In [ ]:
per_user = ratings.groupby('userId').size()
fig, ax = plt.subplots(figsize=(6,4))
per_user.plot(kind='hist', bins=50, ax=ax, color='#2DD4BF')
ax.set_title('Ratings per user')
ax.set_xlabel('# ratings'); plt.tight_layout(); plt.show()
print(per_user.describe())
print(f"Median ratings/user: {per_user.median():.0f}, Max: {per_user.max()}")


## Ratings per movie (popularity long-tail)

In [ ]:
per_movie = ratings.groupby('movieId').size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6,4))
per_movie.reset_index(drop=True).plot(ax=ax, color='#B79CFF')
ax.set_title('Ratings per movie, sorted descending (classic long-tail)')
ax.set_xlabel('movie rank'); ax.set_ylabel('# ratings'); plt.tight_layout(); plt.show()
print(f"Top 10% of movies receive {per_movie.head(int(0.1*len(per_movie))).sum() / per_movie.sum():.1%} of all ratings")


## Genre frequency

In [ ]:
from collections import Counter
genre_counts = Counter(g for gs in movies['genres_list'] for g in gs)
gc = pd.Series(genre_counts).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7,5))
gc.plot(kind='barh', ax=ax, color='#E3B23C')
ax.invert_yaxis(); ax.set_title('Movie count by genre'); plt.tight_layout(); plt.show()


## Ratings over time

In [ ]:
by_year = ratings.groupby(ratings['ts'].dt.year).size()
fig, ax = plt.subplots(figsize=(7,4))
by_year.plot(ax=ax, marker='o', color='#2DD4BF')
ax.set_title('Ratings volume by year'); ax.set_xlabel('year'); plt.tight_layout(); plt.show()


## Sparsity

In [ ]:
n_users, n_movies = ratings['userId'].nunique(), movies['movieId'].nunique()
density = len(ratings) / (n_users * n_movies)
print(f"Users: {n_users:,}  Movies: {n_movies:,}")
print(f"Matrix density: {density:.4%}  (sparsity: {1-density:.4%})")
print("This is why collaborative filtering here MUST use sparse matrices, not dense ones.")


## Missing values / data quality

In [ ]:
print('Null counts:')
for name, df in [('ratings', ratings), ('movies', movies), ('tags', tags), ('links', links)]:
    print(f"  {name}: {df.isnull().sum().sum()} total nulls across {df.shape[1]} columns")
print()
print(f"movies missing year (unparseable title): {movies['year'].isna().sum()}")
print(f"links missing tmdbId: {links['tmdbId'].isna().sum()}")


## Takeaways
- Rating distribution skews positive (people rate what they choose to watch) — relevant when interpreting Popularity's C (catalog mean).
- Both users and movies follow a long-tail distribution — this is exactly why the Popularity model uses Bayesian shrinkage (`src/models/popularity.py`) instead of a raw average, and why Collaborative Filtering must use sparse matrices (`src/models/collaborative.py`).
- A meaningful fraction of movies have no `tmdbId` link or unparseable year — handled gracefully (never crashes) throughout `src/` and `app/`.
